# Image Formation and Capture

## Overview
This notebook explores the fundamentals of image formation - how light from a scene is captured and converted into digital images. We'll demonstrate:
- Image acquisition from files and cameras
- Image properties (dimensions, channels, data types)
- The illumination-reflectance model: I(x,y) = L(x,y) × R(x,y)
- Visualization of image components

## Part 1: Image Acquisition and Properties

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load image
img_path = 'sample.jpeg'
image = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Display basic information
print("=" * 50)
print("IMAGE PROPERTIES")
print("=" * 50)
print(f"Shape (Height, Width, Channels): {image.shape}")
print(f"Data Type: {image.dtype}")
print(f"File Size: {Path(img_path).stat().st_size / 1024:.2f} KB")
print(f"Total Pixels: {image.shape[0]} × {image.shape[1]} = {image.shape[0] * image.shape[1]:,}")
print(f"Memory Size (uncompressed): {image.size * image.itemsize / 1024:.2f} KB")
print(f"Channels: {image.shape[2] if len(image.shape) > 2 else 1}")
print(f"Bit Depth: {image.itemsize * 8} bits per channel")

# Display the image
plt.figure(figsize=(10, 6))
plt.imshow(image_rgb)
plt.title('Original Image')
plt.axis('off')
plt.tight_layout()
plt.show()

## Part 2: Pixel Value Analysis

In [ ]:
# Analyze pixel values statistics
print("\n" + "=" * 50)
print("PIXEL VALUE STATISTICS (BGR Format)")
print("=" * 50)

channels_names = ['Blue', 'Green', 'Red']
channel_stats = {}

for i, name in enumerate(channels_names):
    channel = image[:, :, i]
    stats = {
        'min': channel.min(),
        'max': channel.max(),
        'mean': channel.mean(),
        'std': channel.std(),
        'median': np.median(channel)
    }
    channel_stats[name] = stats
    
    print(f"\n{name} Channel:")
    print(f"  Min:    {stats['min']:.2f}")
    print(f"  Max:    {stats['max']:.2f}")
    print(f"  Mean:   {stats['mean']:.2f}")
    print(f"  Std:    {stats['std']:.2f}")
    print(f"  Median: {stats['median']:.2f}")

## Part 3: Image Formation Model - Illumination and Reflectance

### Physics Model: I(x,y) = L(x,y) × R(x,y)
- **I(x,y)**: Observed pixel intensity
- **L(x,y)**: Illumination (incident light)
- **R(x,y)**: Reflectance (material property, 0-1)

In [ ]:
# Demonstrate the illumination-reflectance model
print("\n" + "=" * 50)
print("ILLUMINATION-REFLECTANCE MODEL")
print("=" * 50)

# Simulate different illumination levels
illuminations = [100, 150, 200]  # Different light levels
reflectances = [0.3, 0.6, 0.9]   # Different material properties

# Create a table of resulting pixel intensities
print("\nPixel Intensity = Illumination × Reflectance\n")
print("Illumination → ")
print(f"{'Reflectance':>12}", end="")
for L in illuminations:
    print(f"{L:>12}", end="")
print()
print("-" * 50)

for R in reflectances:
    print(f"{R:>12.1f}", end="")
    for L in illuminations:
        intensity = L * R
        print(f"{intensity:>12.0f}", end="")
    print()

# Practical examples
print("\n" + "=" * 50)
print("PRACTICAL EXAMPLES")
print("=" * 50)

examples = [
    ("White paper", 150, 0.95),
    ("Gray paper", 150, 0.50),
    ("Dark paper", 150, 0.20),
    ("Same object, bright light", 200, 0.60),
    ("Same object, dim light", 100, 0.60),
]

for description, L, R in examples:
    I = L * R
    print(f"{description:.<30} L={L:>3} × R={R:.2f} = I={I:.0f}")

## Part 4: Channel Decomposition

In [ ]:
# Decompose into separate channels
b, g, r = cv2.split(image)
rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
r_ch, g_ch, b_ch = cv2.split(rgb_image)

# Visualize channels
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Red channel
axes[0, 0].imshow(r_ch, cmap='Reds')
axes[0, 0].set_title('Red Channel')
axes[0, 0].axis('off')

# Green channel
axes[0, 1].imshow(g_ch, cmap='Greens')
axes[0, 1].set_title('Green Channel')
axes[0, 1].axis('off')

# Blue channel
axes[1, 0].imshow(b_ch, cmap='Blues')
axes[1, 0].set_title('Blue Channel')
axes[1, 0].axis('off')

# Original RGB
axes[1, 1].imshow(rgb_image)
axes[1, 1].set_title('Original RGB')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("\nChannel Separation Complete")
print(f"Red channel range: [{r_ch.min()}, {r_ch.max()}]")
print(f"Green channel range: [{g_ch.min()}, {g_ch.max()}]")
print(f"Blue channel range: [{b_ch.min()}, {b_ch.max()}]")

## Part 5: Histogram Analysis

In [ ]:
# Calculate and visualize histograms
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['red', 'green', 'blue']
channels = [r_ch, g_ch, b_ch]
channel_names = ['Red', 'Green', 'Blue']

for idx, (channel, color, name) in enumerate(zip(channels, colors, channel_names)):
    hist = cv2.calcHist([channel], [0], None, [256], [0, 256])
    
    axes[idx].plot(hist, color=color, linewidth=2)
    axes[idx].set_title(f'{name} Channel Histogram')
    axes[idx].set_xlabel('Pixel Intensity')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)
    axes[idx].set_xlim([0, 256])

plt.tight_layout()
plt.show()

print("Histogram Analysis:")
for channel, name in zip(channels, channel_names):
    hist = cv2.calcHist([channel], [0], None, [256], [0, 256])
    print(f"\n{name} Channel:")
    print(f"  Most common intensity: {np.argmax(hist)}")
    print(f"  Peak frequency: {hist.max():.0f}")
    print(f"  Zero pixels (intensity=0): {hist[0][0]:.0f}")
    print(f"  Max pixels (intensity=255): {hist[255][0]:.0f}")

## Part 6: Pixel Indexing and Access

In [ ]:
# Demonstrate pixel access
print("\n" + "=" * 50)
print("PIXEL INDEXING AND ACCESS")
print("=" * 50)

# Access specific pixels
h, w = image.shape[:2]
test_points = [
    (0, 0, "Top-left corner"),
    (0, w-1, "Top-right corner"),
    (h-1, 0, "Bottom-left corner"),
    (h-1, w-1, "Bottom-right corner"),
    (h//2, w//2, "Center"),
]

print("\nSample Pixel Values (BGR format):\n")
for y, x, label in test_points:
    pixel_bgr = image[y, x]
    pixel_rgb = rgb_image[y, x]
    print(f"Position {label:.<20} ({y:>4}, {x:>4}): BGR={pixel_bgr}, RGB={pixel_rgb}")

# Create a visual map showing pixel positions
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(rgb_image)

# Mark test points
for y, x, label in test_points:
    ax.plot(x, y, 'ro', markersize=10)
    ax.text(x+10, y+10, label, color='white', fontsize=9, 
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

ax.set_title('Pixel Positions')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 7: Bit Depth and Dynamic Range

In [ ]:
# Demonstrate different bit depths
print("\n" + "=" * 50)
print("BIT DEPTH AND DYNAMIC RANGE")
print("=" * 50)

# Convert to different data types
image_float32 = image.astype(np.float32) / 255.0  # Normalize to 0-1
image_float64 = image.astype(np.float64) / 255.0
image_uint16 = image.astype(np.uint16) * 257     # Scale to 16-bit

print("\nData Type Comparison:")
print(f"Original (uint8):   dtype={image.dtype}, range=[0, 255], itemsize={image.itemsize} bytes")
print(f"Float32:            dtype={image_float32.dtype}, range=[0.0, 1.0], itemsize={image_float32.itemsize} bytes")
print(f"Float64:            dtype={image_float64.dtype}, range=[0.0, 1.0], itemsize={image_float64.itemsize} bytes")
print(f"Uint16:             dtype={image_uint16.dtype}, range=[0, 65535], itemsize={image_uint16.itemsize} bytes")

print("\nBit Depth Analysis:")
bit_depths = [
    (8, "0-255", "Image display"),
    (16, "0-65535", "Medical imaging"),
    (32, "0.0-1.0 (float)", "Processing, calculations"),
    (64, "0.0-1.0 (double)", "Scientific computing"),
]

for bits, range_str, use_case in bit_depths:
    levels = 2 ** bits
    print(f"{bits:>2}-bit: {range_str:>15} ({levels:>10,} levels) - {use_case}")

# Memory usage comparison
print("\nMemory Usage for Same Image:")
base_size = image.size
print(f"uint8:   {base_size * 1 / 1024:.2f} KB")
print(f"uint16:  {base_size * 2 / 1024:.2f} KB")
print(f"float32: {base_size * 4 / 1024:.2f} KB")
print(f"float64: {base_size * 8 / 1024:.2f} KB")

## Part 8: Image Resizing and Interpolation

In [ ]:
# Demonstrate different interpolation methods
print("\n" + "=" * 50)
print("INTERPOLATION METHODS FOR RESIZING")
print("=" * 50)

# Resize to smaller size
h, w = image.shape[:2]
new_h, new_w = h // 2, w // 2

interpolations = [
    (cv2.INTER_NEAREST, "Nearest Neighbor"),
    (cv2.INTER_LINEAR, "Bilinear"),
    (cv2.INTER_CUBIC, "Bicubic"),
    (cv2.INTER_AREA, "Area-based (RESAMPLING)"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (method, name) in enumerate(interpolations):
    resized = cv2.resize(rgb_image, (new_w, new_h), interpolation=method)
    axes[idx].imshow(resized)
    axes[idx].set_title(f'{name} (50% downscale)')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"\nOriginal size: {w} × {h}")
print(f"Resized size: {new_w} × {new_h}")
print("\nInterpolation methods:")
print("- Nearest Neighbor: Fast, blocky (good for pixel art)")
print("- Bilinear: Linear interpolation, balance of speed and quality")
print("- Bicubic: Better quality, slower, smoother results")
print("- Area-based: Best for downsampling, good quality")

## Part 9: Summary

### Key Concepts Covered:
1. **Image Formation**: Light → Object → Camera → Digital Image
2. **Image Properties**: Resolution, channels, bit depth, data type
3. **Pixel Values**: RGB components, intensity ranges, statistics
4. **Illumination-Reflectance Model**: I(x,y) = L(x,y) × R(x,y)
5. **Channel Operations**: Decomposition, individual channel analysis
6. **Histograms**: Distribution of pixel intensities
7. **Pixel Access**: Indexing and coordinate systems
8. **Data Types**: uint8, uint16, float32, float64
9. **Resizing**: Interpolation methods for image scaling

### Next Steps:
- Proceed to Unit 1: Pixel and Grayscale (01_Pixel_and_Grayscale.ipynb)
- Learn about color space conversions and transformations